# Matrix Transformation

> Matrix 是一台重塑空间的机器。理解它对每个点做了什么，你就理解了整个 transformation。

**类型：** 动手实现
**语言：** Python, Julia
**前置要求：** Phase 1, Lesson 01-02（线性代数直觉、Vector 与 Matrix 运算）
**时间：** ~75 分钟

## 术语对照

- eigenvalue，特征值
- eigenvector，特征向量
- transformation，变换
- rotation，旋转
- scaling，缩放
- shearing，剪切
- reflection，反射
- covariance matrix，协方差矩阵
- determinant，行列式
- PCA，主成分分析
- spectral clustering，谱聚类
## 关键术语

| 术语 | 人们怎么说 | 实际含义 |
|------|-----------|---------|
| 旋转 matrix | "转动东西" | 一个正交 matrix，将点沿圆弧移动，同时保持距离和角度。determinant 始终为 1。 |
| 缩放 matrix | "把东西变大" | 一个对角 matrix，沿每条轴独立拉伸或压缩。determinant 是各缩放因子的乘积。 |
| 剪切 matrix | "倾斜东西" | 一个将一个坐标按比例偏移到另一个坐标的 matrix，将矩形变成平行四边形。determinant 为 1。 |
| 反射 | "镜像东西" | 一个将空间关于某条轴或某个平面翻转的 matrix。determinant 为 -1。 |
| 组合 | "做两件事" | 将 transformation matrix 相乘以串联操作。顺序很重要：B @ A 表示先应用 A，再应用 B。 |
| Eigenvector | "特殊方向" | 一个 matrix 只缩放、从不旋转的方向。Transformation 的指纹。 |
| Eigenvalue | "拉伸了多少倍" | Matrix 对其 eigenvector 进行缩放的标量因子。可以为负（翻转）或复数（旋转）。 |
| 特征分解 | "拆开 matrix" | 将 matrix 写为 V @ D @ V^(-1)，将其分解为基本缩放方向和大小。 |
| Determinant | "从 matrix 得出的一个数" | Transformation 缩放面积（2D）或体积（3D）的因子。零意味着 transformation 不可逆。 |
| 特征方程 | "eigenvalue 的来源" | det(A - lambda * I) = 0。其根即为 eigenvalue 的多项式。 |

## 扩展阅读

- [3Blue1Brown：线性 Transformations](https://www.3blue1brown.com/lessons/linear-transformations) —— matrix 如何重塑空间的视觉直觉
- [3Blue1Brown：Eigenvector 与 Eigenvalue](https://www.3blue1brown.com/lessons/eigenvalues) —— eigenvector 几何含义的最佳视觉解释
- [MIT 18.06 Lecture 21：Eigenvalue 与 Eigenvector](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/) —— Gilbert Strang 的经典讲授

## 学习目标

- 构造旋转、缩放、剪切和反射 matrix 并将其应用于 2D 和 3D 点
- 通过矩阵乘法组合多个 transformation，并验证顺序的重要性
- 从特征方程计算 2×2 matrix 的 eigenvalue 和 eigenvector
- 解释为什么 eigenvalue 决定了 PCA 方向、RNN 稳定性和谱聚类行为

## 问题

你读到 PCA 时看到"求 covariance matrix 的 eigenvector"。你读到模型稳定性时看到"检查是否所有 eigenvalue 的 magnitude 小于 1"。你读到数据增强时看到"应用一个随机旋转"。在理解 matrix 在几何上对空间做了什么之前，这些都不 make sense。

Matrix 不只是数字表格。它们是空间机器。一个旋转 matrix 转动点。一个缩放 matrix 拉伸点。一个剪切 matrix 倾斜点。神经网络对数据施加的每一个 transformation 都是这些操作之一或它们的组合。这节课让这些操作变得具体。

## 概念

### Transformation 即 matrix

每个 2D 线性 transformation 都可以写成一个 2×2 matrix。这个 matrix 精确告诉了你 basis vector [1, 0] 和 [0, 1] 会落在哪里。其他一切都随之确定。

```mermaid
graph LR
    subgraph Before["标准 Basis"]
        e1["e1 = [1, 0]（沿 x 轴）"]
        e2["e2 = [0, 1]（沿 y 轴）"]
    end
    subgraph Transform["Matrix M"]
        M["M = 各列即新的 basis vector"]
    end
    subgraph After["Transformation M 之后"]
        e1p["e1' = 新的 x-basis"]
        e2p["e2' = 新的 y-basis"]
    end
    e1 --> M --> e1p
    e2 --> M --> e2p


## 概念

### Transformation 即 Matrix

每个 2D 线性变换都可以写成一个 2×2 矩阵。矩阵的**第一列是 $[1,0]$ 变换后的去向，第二列是 $[0,1]$ 的去向**。知道这两列，就知道所有点的去向。

### 四种基本变换

| 变换 | 2×2 矩阵 | det | 效果 |
|------|----------|-----|------|
| **旋转 $\theta$** | $\begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$ | 1 | 沿圆弧移动，保距保角 |
| **缩放 $(s_x, s_y)$** | $\begin{bmatrix} s_x & 0 \\ 0 & s_y \end{bmatrix}$ | $s_x \cdot s_y$ | 沿轴独立拉伸 |
| **剪切 x $(k)$** | $\begin{bmatrix} 1 & k \\ 0 & 1 \end{bmatrix}$ | 1 | 矩形→平行四边形 |
| **关于 y 反射** | $\begin{bmatrix} -1 & 0 \\ 0 & 1 \end{bmatrix}$ | -1 | 镜像翻转 |

### 旋转

2D 旋转（角度 $\theta$）保持距离和角度不变。它将每个点沿圆弧移动。

```mermaid
graph LR
    subgraph Before["旋转前"]
        A["A(2, 1)"]
        B["B(0, 2)"]
    end
    subgraph Rot["旋转 45°"]
        R["R(θ) = [[cos θ, -sin θ], [sin θ, cos θ]]"]
    end
    subgraph After["旋转后"]
        Ap["A'(0.71, 2.12)"]
        Bp["B'(-1.41, 1.41)"]
    end
    A --> R --> Ap
    B --> R --> Bp
```

在 3D 中，你绕轴旋转。每条轴有自己的旋转 matrix：

```
Rz(theta) = | cos  -sin  0 |     绕 z 轴旋转
            | sin   cos  0 |     (x-y 平面旋转，z 保持不变)
            |  0     0   1 |

Rx(theta) = | 1   0     0    |   绕 x 轴旋转
            | 0  cos  -sin   |   (y-z 平面旋转，x 保持不变)
            | 0  sin   cos   |

Ry(theta) = |  cos  0  sin |     绕 y 轴旋转
            |   0   1   0  |     (x-z 平面旋转，y 保持不变)
            | -sin  0  cos |
```

### 缩放

缩放沿每条轴独立地拉伸或压缩。

```mermaid
graph LR
    subgraph Before["缩放前"]
        A["A(2, 1)"]
        B["B(0, 2)"]
    end
    subgraph Scale["缩放 sx=2, sy=0.5"]
        S["S = [[2, 0], [0, 0.5]]"]
    end
    subgraph After["缩放后"]
        Ap["A'(4, 0.5)"]
        Bp["B'(0, 1)"]
    end
    A --> S --> Ap
    B --> S --> Bp
```

### 剪切

剪切倾斜一条轴同时保持另一条固定。它将矩形变成平行四边形。

```mermaid
graph LR
    subgraph Before["剪切前"]
        A["A(1, 0)"]
        B["B(0, 1)"]
    end
    subgraph Shear["沿 x 剪切, k=1"]
        Sh["Shx = [[1, k], [0, 1]]"]
    end
    subgraph After["剪切后"]
        Ap["A(1, 0) 不变"]
        Bp["B'(1, 1) 被移动"]
    end
    A --> Sh --> Ap
    B --> Sh --> Bp
```

剪切 matrix：
- `Shx = [[1, k], [0, 1]]` 将 x 偏移 k×y
- `Shy = [[1, 0], [k, 1]]` 将 y 偏移 k×x

### 反射

反射将点关于某条轴或某条线镜像翻转。

```mermaid
graph LR
    subgraph Before["反射前"]
        A["A(2, 1)"]
    end
    subgraph Reflect["关于 y 轴反射"]
        R["[[-1, 0], [0, 1]]"]
    end
    subgraph After["反射后"]
        Ap["A'(-2, 1)"]
    end
    A --> R --> Ap
```

反射 matrix：
- 关于 y 轴反射：`[[-1, 0], [0, 1]]`
- 关于 x 轴反射：`[[1, 0], [0, -1]]`

### 组合：顺序至关重要

先应用 transformation A 再应用 B，等同于将它们的 matrix 相乘：`result = B @ A @ point`。

$S \cdot R \neq R \cdot S$。先旋转再缩放 ≠ 先缩放再旋转。矩阵乘法不可交换。

```mermaid
graph LR
    subgraph Path1["路径 1：旋转 90° → 缩放 (2, 0.5)"]
        P1["(1, 0)"] -->|"旋转 90°"| P2["(0, 1)"] -->|"缩放"| P3["(0, 0.5)"]
    end
```

组合结果：`S @ R = [[0, -2], [0.5, 0]]`

```mermaid
graph LR
    subgraph Path2["路径 2：缩放 (2, 0.5) → 旋转 90°"]
        Q1["(1, 0)"] -->|"缩放"| Q2["(2, 0)"] -->|"旋转 90°"| Q3["(0, 2)"]
    end
```

组合结果：`R @ S = [[0, -0.5], [2, 0]]`

不同的结果。矩阵乘法不可交换。

### Eigenvalue 与 Eigenvector

大多数 vector 在被 matrix 作用时会改变方向。Eigenvector 是特殊的：matrix 只缩放它们，从不旋转它们。

$$A \cdot \mathbf{v} = \lambda \cdot \mathbf{v}$$

- **Eigenvector $\mathbf{v}$**：方向不变、只被拉伸的向量（$\mathbf{v} \neq \mathbf{0}$）
- **Eigenvalue $\lambda$**：拉伸的倍数

> 两者必须配对：$\lambda=3$ 告诉你"有东西拉伸了 3 倍"，$\mathbf{v}=[1,1]$ 告诉你"是 45° 这个方向"。

**最直观的例子：**

$$A = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$$

- $\mathbf{v}_1 = [1,1]$：$A \cdot [1,1] = [3,3] = 3 \cdot [1,1]$ → 45° 方向，拉伸 3 倍 ✅
- $\mathbf{v}_2 = [1,-1]$：$A \cdot [1,-1] = [1,-1] = 1 \cdot [1,-1]$ → -45° 方向，不变 ✅
- $[1,0]$：$A \cdot [1,0] = [2,1]$ → 方向变了 ❌ 不是 eigenvector

**几何：** 这个矩阵把单位圆拉成椭圆。长轴方向 $[1,1]$（拉伸 3 倍），短轴方向 $[1,-1]$（不变）。这两个轴就是 eigenvector。

### 怎么求？——特征方程

从 $A\mathbf{v} = \lambda\mathbf{v}$ 出发：

$$A\mathbf{v} - \lambda I\mathbf{v} = \mathbf{0} \quad\Rightarrow\quad (A - \lambda I)\mathbf{v} = \mathbf{0}$$

要 $\mathbf{v} \neq \mathbf{0}$，必须 $A - \lambda I$ 奇异 → $\det(A - \lambda I) = 0$。

**$A-\lambda I$ 是一个"探测器"**——我们故意让它压缩一维（行列式=0），只为找到那个让条件成立的 $\lambda$。找到后探测器就扔了，真正要分析的是 $A$ 本身。

对 2×2 矩阵展开：

$$\det\begin{bmatrix} a-\lambda & b \\ c & d-\lambda \end{bmatrix} = \lambda^2 - (a+d)\lambda + (ad-bc) = 0$$

$$\lambda = \frac{\text{trace} \pm \sqrt{\text{trace}^2 - 4\det}}{2}$$

### 三种判别式情况

| 判别式 | 结果 | 含义 | 例子 |
|--------|------|------|------|
| > 0 | 两个不同实数 $\lambda$ | 沿两个独立方向拉伸 | $A=[2,1;1,2]$ |
| = 0 | 一个重复实数 $\lambda$ | 可能只有一个方向 | $A=[2,0;0,2]$ |
| < 0 | 一对共轭复数 $\lambda$ | 纯旋转，无实数方向 | $R(90°)$ |

### 找 $\mathbf{v}$：代回解方程

把每个 $\lambda$ 代回 $(A - \lambda I)\mathbf{v} = \mathbf{0}$，解出 $\mathbf{v}$。整条线上的向量都是 eigenvector，通常归一化到长度 1 作为代表。

### 特征分解

$$A = V \cdot D \cdot V^{-1}$$

$V$ 的列是 eigenvector，$D$ 的对角线是 eigenvalue。**含义：** 旋转到 eigen 坐标系 → 沿每轴独立缩放 → 旋转回来。

### 为什么 ML 依赖它们？

| 应用 | eigenvector | eigenvalue |
|------|------------|------------|
| **PCA 降维** | 主成分方向（方差最大方向） | 该方向的方差大小 |
| **RNN 稳定性** | — | $|\lambda|>1$ → 梯度爆炸；$|\lambda|<1$ → 梯度消失 |
| **谱聚类** | 图的社区结构 | 连通性强弱 |

### Determinant = 体积缩放因子

| det 值 | 含义 |
|--------|------|
| $=1$ | 面积/体积不变（旋转、剪切）|
| $=0$ | 空间坍缩到更低维度（奇异矩阵，不可逆）|
| $=-1$ | 面积不变但方向翻转（反射）|

对组合变换：$\det(B \cdot A) = \det(B) \cdot \det(A)$

---

## Build It + Use It + 练习

> 本节代码直接取自官方 `en.md` 的 Build It / Use It section，逐行添加中文注释讲解。
> 先读懂完整代码（Build It + Use It），然后用下方的**分层练习**检验理解：先做基础题，再按需要挑战进阶题。

### 第 1 步：从零实现 transformation matrix（Build It）

直接照搬 `en.md` Build It Step 1 的代码。每个逻辑块上方添加中文注释解释它在做什么。

In [85]:
import math

# ----------------------------------------------------------------
# 2D 旋转矩阵：将点绕原点旋转 theta 弧度
# 矩阵形式：[[cosθ, -sinθ], [sinθ, cosθ]]
# 第一列 = (1,0) 旋转后的位置，第二列 = (0,1) 旋转后的位置
# ----------------------------------------------------------------
def rotation_2d(theta):
    c, s = math.cos(theta), math.sin(theta)
    return [[c, -s], [s, c]]

# ----------------------------------------------------------------
# 2D 缩放矩阵：沿 x 轴拉伸 sx 倍，沿 y 轴拉伸 sy 倍
# 矩阵形式：[[sx, 0], [0, sy]]  —— 对角矩阵，对角元即缩放因子
# ----------------------------------------------------------------
def scaling_2d(sx, sy):
    return [[sx, 0], [0, sy]]

# ----------------------------------------------------------------
# 2D 剪切矩阵：将 x 坐标偏移 kx*y，将 y 坐标偏移 ky*x
# kx 控制"水平倾斜"，ky 控制"垂直倾斜"
# 矩阵形式：[[1, kx], [ky, 1]]  —— 对角元为 1（保持面积不变）
# ----------------------------------------------------------------
def shearing_2d(kx, ky):
    return [[1, kx], [ky, 1]]

# ----------------------------------------------------------------
# 反射矩阵：关于 x 轴或 y 轴镜像翻转
# 关于 x 轴反射：y 坐标取反
# 关于 y 轴反射：x 坐标取反
# ----------------------------------------------------------------
def reflection_x():
    return [[1, 0], [0, -1]]

def reflection_y():
    return [[-1, 0], [0, 1]]

# ----------------------------------------------------------------
# 矩阵×向量乘法：对每一行 i，计算该行与向量的点积
# result[i] = sum_j matrix[i][j] * vector[j]
# ----------------------------------------------------------------
def mat_vec_mul(matrix, vector):
    return [
        sum(matrix[i][j] * vector[j] for j in range(len(vector)))
        for i in range(len(matrix))
    ]

# ----------------------------------------------------------------
# 矩阵×矩阵乘法：result[i][j] = A 的第 i 行 · B 的第 j 列
# A 的列数必须等于 B 的行数
# ----------------------------------------------------------------
def mat_mul(a, b):
    rows_a, cols_b = len(a), len(b[0])
    cols_a = len(a[0])
    return [
        [sum(a[i][k] * b[k][j] for k in range(cols_a)) for j in range(cols_b)]
        for i in range(rows_a)
    ]

# ----------------------------------------------------------------
# 测试：对具体点应用各变换，观察效果
# ----------------------------------------------------------------
point = [1.0, 0.0]
angle = math.pi / 4

rotated = mat_vec_mul(rotation_2d(angle), point)
print(f"将 (1,0) 旋转 45°：({rotated[0]:.4f}, {rotated[1]:.4f})")

scaled = mat_vec_mul(scaling_2d(2, 3), [1.0, 1.0])
print(f"将 (1,1) 缩放 (2,3)：({scaled[0]:.1f}, {scaled[1]:.1f})")

sheared = mat_vec_mul(shearing_2d(1, 0), [1.0, 1.0])
print(f"将 (1,1) 剪切 kx=1：({sheared[0]:.1f}, {sheared[1]:.1f})")

reflected = mat_vec_mul(reflection_y(), [2.0, 1.0])
print(f"将 (2,1) 关于 y 反射：({reflected[0]:.1f}, {reflected[1]:.1f})")

将 (1,0) 旋转 45°：(0.7071, 0.7071)
将 (1,1) 缩放 (2,3)：(2.0, 3.0)
将 (1,1) 剪切 kx=1：(2.0, 1.0)
将 (2,1) 关于 y 反射：(-2.0, 1.0)


### 第 2 步：组合变换（Build It）

直接照搬 `en.md` Build It Step 2。**核心知识点：矩阵乘法不可交换**——先旋转再缩放 ≠ 先缩放再旋转。

In [86]:
# ----------------------------------------------------------------
# 组合变换：先旋转 90° 再缩放 (2, 0.5) → S @ R
# 矩阵乘法从右往左读：mat_mul(S, R) 表示先应用 R，再应用 S
# ----------------------------------------------------------------
R = rotation_2d(math.pi / 2)
S = scaling_2d(2, 0.5)

rotate_then_scale = mat_mul(S, R)   # S @ R：先旋转，再缩放
scale_then_rotate = mat_mul(R, S)   # R @ S：先缩放，再旋转

point = [1.0, 0.0]
result1 = mat_vec_mul(rotate_then_scale, point)
result2 = mat_vec_mul(scale_then_rotate, point)

print(f"先旋转 90° 再缩放：({result1[0]:.2f}, {result1[1]:.2f})")
print(f"先缩放再旋转 90°：({result2[0]:.2f}, {result2[1]:.2f})")
# 关键结论：两者结果不同！矩阵乘法不可交换
print(f"相同吗？{result1 == result2}")

先旋转 90° 再缩放：(0.00, 0.50)
先缩放再旋转 90°：(0.00, 2.00)
相同吗？False


### 第 3 步：从零计算 eigenvalue（Build It）

对于 2×2 矩阵 `[[a, b], [c, d]]`，eigenvalue 解特征方程：

$$\lambda^2 - (a+d)\lambda + (ad-bc) = 0$$

其中 trace = a+d，determinant = ad-bc。

Matrix: [[2, 1], [1, 2]]
Eigenvalue: 3.0000, 1.0000
  lambda=3.0, v=[0.7071, 0.7071]
    A@v = [2.1213, 2.1213]
    l*v = [2.1213, 2.1213]
  lambda=1.0, v=[0.7071, -0.7071]
    A@v = [0.7071, -0.7071]
    l*v = [0.7071, -0.7071]


In [ ]:
# ----------------------------------------------------------------
# 2×2 矩阵的 eigenvalue 计算
# 特征方程：λ² - trace·λ + det = 0
# 判别式 > 0：两个不同实数根
# 判别式 = 0：一个重根
# 判别式 < 0：一对共轭复根（表示旋转，无实数方向）
# ----------------------------------------------------------------
def eigenvalues_2x2(matrix):
    a, b = matrix[0]
    c, d = matrix[1]
    trace = a + d                       # 对角元之和
    det = a * d - b * c                 # ad - bc
    discriminant = trace ** 2 - 4 * det # 判别式 Δ
    if discriminant < 0:                # 复数根：实部 + 虚部
        real = trace / 2
        imag = (-discriminant) ** 0.5 / 2 # 复数不能** 0.5，需要变成正数再开根
        return (complex(real, imag), complex(real, -imag))
    sqrt_disc = discriminant ** 0.5     # 实数根
    return ((trace + sqrt_disc) / 2, (trace - sqrt_disc) / 2)

# ----------------------------------------------------------------
# Eigenvector 计算：对已知 λ，解 (A - λI)v = 0
# 注意：eigenvector 不唯一（整条线都是），这里返回归一化版本
# ----------------------------------------------------------------
def eigenvector_2x2(matrix, eigenvalue):
    a, b = matrix[0]
    c, d = matrix[1]
    # 选择非零分量来构造 v，避免除以零
    if abs(b) > 1e-10:
        v = [b, eigenvalue - a]          # 用 b 构造
    elif abs(c) > 1e-10:
        v = [eigenvalue - d, c]          # 用 c 构造
    else:
        if abs(a - eigenvalue) < 1e-10:  # 对角矩阵情况
            v = [1, 0]
        else:
            v = [0, 1]
    mag = (v[0] ** 2 + v[1] ** 2) ** 0.5  # 归一化到单位长度
    return [v[0] / mag, v[1] / mag]

# ----------------------------------------------------------------
# 测试：A = [[2, 1], [1, 2]]，已知 eigenvalue 为 3 和 1
# ----------------------------------------------------------------
A = [[2, 1], [1, 2]]
vals = eigenvalues_2x2(A)
print(f"Matrix: {A}")
print(f"Eigenvalue: {vals[0]:.4f}, {vals[1]:.4f}")

for val in vals:
    vec = eigenvector_2x2(A, val)
    # 为了检验是否 Av=λv，分别算出 Av 和 λv，两者相等说明 vec 确实是特征向量
    result = mat_vec_mul(A, vec)
    scaled = [val * vec[0], val * vec[1]]
    print(f"  lambda={val:.1f}, v={[round(x,4) for x in vec]}")
    print(f"    A@v = {[round(x,4) for x in result]}") # round(x, 4)把数字 x 四舍五入到小数点后 4 位
    print(f"    l*v = {[round(x,4) for x in scaled]}")
    # A@v 和 λv 应该相等——这就是 eigenvalue 的定义

### 第 4 步：Determinant 作为体积缩放因子（Build It）

Determinant 的值揭示变换的本质：

| det | 含义 |
|-----|------|
| = 1 | 保面积（旋转、剪切） |
| = 0 | 空间坍缩（奇异、不可逆） |
| = -1 | 保面积但翻转方向（反射） |

In [88]:
# ----------------------------------------------------------------
# 2×2 determinant：ad - bc
# 几何含义：变换后单位正方形的面积缩放因子
# ----------------------------------------------------------------
def det_2x2(matrix):
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]

# 验证各种变换的 determinant
print(f"det(旋转 45°) = {det_2x2(rotation_2d(math.pi/4)):.4f}")   # 应为 1
print(f"det(缩放 2,3)   = {det_2x2(scaling_2d(2, 3)):.1f}")       # 应为 6 = 2×3
print(f"det(剪切 kx=1)  = {det_2x2(shearing_2d(1, 0)):.1f}")      # 应为 1
print(f"det(关于 y 反射) = {det_2x2(reflection_y()):.1f}")          # 应为 -1

# 奇异矩阵示例：第二列是第一列的 2 倍 → 空间坍缩为一条线
singular = [[1, 2], [2, 4]]
print(f"det(奇异)        = {det_2x2(singular):.1f}")
print("奇异 matrix：列成比例，空间坍缩为一条线。")

det(旋转 45°) = 1.0000
det(缩放 2,3)   = 6.0
det(剪切 kx=1)  = 1.0
det(关于 y 反射) = -1.0
det(奇异)        = 0.0
奇异 matrix：列成比例，空间坍缩为一条线。


### Use It：NumPy 等价实现

直接照搬 `en.md` Use It section。NumPy 用 C/Fortran 实现，比我们的纯 Python 快 ~100 倍，但底层算法是一回事。

In [89]:
import numpy as np

# ----------------------------------------------------------------
# NumPy 旋转：np.array 直接构造矩阵，@ 运算符做矩阵乘法
# ----------------------------------------------------------------
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

point = np.array([1.0, 0.0])
print(f"将 (1,0) 旋转 45°：{R @ point}")

# ----------------------------------------------------------------
# 组合变换：np.diag 创建对角缩放矩阵，@ 串联操作
# ----------------------------------------------------------------
S = np.diag([2.0, 3.0])
composed = S @ R                    # 先旋转再缩放
print(f"旋转 45° 后缩放 (2,3)：{composed @ point}")

# ----------------------------------------------------------------
# NumPy eigenvalue：np.linalg.eig 一行搞定
# 返回 (eigenvalue 数组, eigenvector 矩阵（各列）)
# ----------------------------------------------------------------
A = np.array([[2, 1], [1, 2]], dtype=float)
eigenvalues, eigenvectors = np.linalg.eig(A) # 拿到特征值和特征向量
print(f"\nEigenvalue: {eigenvalues}")
print(f"Eigenvector（各列）：\n{eigenvectors}")

for i in range(len(eigenvalues)): # 有几个特征值说明有几个特征向量
    v = eigenvectors[:, i] # 取出第 i 列 → 第 i 个 eigenvector
    lam = eigenvalues[i]
    print(f"  A @ v{i} = {A @ v}, lambda * v{i} = {lam * v}")

# ----------------------------------------------------------------
# NumPy determinant：np.linalg.det
# ----------------------------------------------------------------
print(f"\ndet(R) = {np.linalg.det(R):.4f}")
print(f"det(S) = {np.linalg.det(S):.1f}")

# ----------------------------------------------------------------
# 特征分解验证：A = V @ D @ V^(-1)
# 旋转到 eigen 坐标系 → 沿各轴独立缩放 → 旋转回来
# ----------------------------------------------------------------
B = np.array([[3, 1], [0, 2]], dtype=float)
vals, vecs = np.linalg.eig(B)
D = np.diag(vals)
V = vecs
reconstructed = V @ D @ np.linalg.inv(V)
print(f"\n特征分解 A = V @ D @ V^-1：")
print(f"原始 Matrix：\n{B}")
print(f"重建 Matrix：\n{reconstructed}")

将 (1,0) 旋转 45°：[0.70710678 0.70710678]
旋转 45° 后缩放 (2,3)：[1.41421356 2.12132034]

Eigenvalue: [3. 1.]
Eigenvector（各列）：
[[ 0.70710678 -0.70710678]
 [ 0.70710678  0.70710678]]
  A @ v0 = [2.12132034 2.12132034], lambda * v0 = [2.12132034 2.12132034]
  A @ v1 = [-0.70710678  0.70710678], lambda * v1 = [-0.70710678  0.70710678]

det(R) = 1.0000
det(S) = 6.0

特征分解 A = V @ D @ V^-1：
原始 Matrix：
[[3. 1.]
 [0. 2.]]
重建 Matrix：
[[3. 1.]
 [0. 2.]]


### NumPy 3D 旋转

直接照搬 `en.md` 的 3D 旋转代码。在 3D 中你绕轴旋转（而不是绕原点），每条轴有专属的旋转矩阵。

In [90]:
# ----------------------------------------------------------------
# 3D 旋转：绕 z 轴（x-y 平面旋转，z 不变）
# 3D 旋转矩阵是 2D 旋转矩阵的"嵌入"——在对应平面旋转，第三轴保持不变
# ----------------------------------------------------------------
def rotation_3d_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def rotation_3d_x(theta): # 绕哪个轴旋转，哪个轴不变
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

point_3d = np.array([1.0, 0.0, 0.0])
rotated_z = rotation_3d_z(np.pi / 2) @ point_3d
rotated_x = rotation_3d_x(np.pi / 2) @ point_3d

print(f"\n3D 点：{point_3d}")
print(f"绕 z 轴旋转 90°：{np.round(rotated_z, 4)}")
print(f"绕 x 轴旋转 90°：{np.round(rotated_x, 4)}")


3D 点：[1. 0. 0.]
绕 z 轴旋转 90°：[0. 1. 0.]
绕 x 轴旋转 90°：[1. 0. 0.]


---

## 练习（分层：先会做，再会解释，最后再挑战）

> 这组练习不再追求一步到位。你按顺序做：
>
> 1. **基础必做**：会把一个 matrix 应用到一组点上，并观察点的位置怎么变。
> 2. **进阶选做**：比较两种组合顺序，亲手看到“先做 A 再做 B”和“先做 B 再做 A”不一样。
> 3. **挑战选做**：用 eigenvalue 做一个很小的行为判断，只要求会用，不要求你完整重写特征分解。
>
> 每题都有三部分：题目说明、你要补的函数、测试 cell。测试 cell 不要改；如果测试通过，说明这题已经达标。
>
> 建议：先只做基础必做。基础题做顺了，再碰进阶题。挑战题可以先跳过，等你觉得 eigenvalue 更熟以后再回来。

### 练习 1（基础必做）：把 matrix 应用到一组点上

这一题只练一个最基本动作：**给定一个 matrix 和一组点，算出每个点变换后的位置**。

你不用推公式，也不用判断复杂性质。只要把前面 Build It 里已经见过的 `mat_vec_mul(matrix, vector)` 用在多个点上。

**你要实现两个函数：**

```text
apply_matrix_to_points(matrix, points) -> transformed_points
edge_lengths(points) -> lengths
```

**步骤提示：**

1. `apply_matrix_to_points`：新建一个空列表。
2. 逐个取出 `points` 里的点 `p`。
3. 用 `mat_vec_mul(matrix, p)` 得到变换后的点。
4. 把结果放进新列表，最后返回。
5. `edge_lengths`：把四个点看成一个四边形，依次计算相邻点之间的距离。
6. 最后一条边是“最后一个点到第一个点”。

**你应该观察到：**

- 旋转后的正方形会转方向，但边长还是一样。
- 缩放后的正方形边长会变。
- 剪切后的形状会斜掉。

In [91]:
import math


def apply_matrix_to_points(matrix, points):
    """把同一个 matrix 应用到一组 2D 点上。

    matrix: 2×2 matrix，例如 [[a, b], [c, d]]
    points: 多个点，例如 [[0, 0], [1, 0], [1, 1]]
    返回：变换后的点列表，顺序和 points 保持一致。
    """
    return [mat_vec_mul(matrix, point) for point in points]
    


def edge_lengths(points):
    """计算四边形四条边的长度。

    points: 四个点，按顺时针或逆时针顺序排列。
    返回：四条边长 [边0, 边1, 边2, 边3]。
    """
    # 一个点里面的两个数字对应相减后，相当于构造了一个新向量，求新向量的模长=两个的点的两个数字分别相减分别平方相加后开根号
    lengths = [] # 收集结果
    for i in range(len(points)):
        p1 = points[i]
        p2 = points[(i+1)%4]
        dx = p1[0] - p2[0]
        dy = p1[1] - p2[1]
        lengths.append(((dx**2) + (dy**2)) **0.5)
    return lengths

In [92]:
# === 练习 1 测试（不要修改）===
# 先跑这个测试。通过后，再看下面打印出来的点和边长。

square = [[0, 0], [1, 0], [1, 1], [0, 1]]
R45 = rotation_2d(math.pi / 4)
S = scaling_2d(2, 0.5)
Sh = shearing_2d(0.5, 0)

rotated = apply_matrix_to_points(R45, square)
scaled = apply_matrix_to_points(S, square)
sheared = apply_matrix_to_points(Sh, square)

assert len(rotated) == 4, "应该返回 4 个变换后的点"
assert all(len(p) == 2 for p in rotated), "每个点都应该有 2 个坐标"

# 检查几个明确的点，确认 matrix 确实作用到了每个点上。
assert [round(x, 4) for x in rotated[1]] == [0.7071, 0.7071]
assert [round(x, 4) for x in scaled[2]] == [2, 0.5]
assert [round(x, 4) for x in sheared[2]] == [1.5, 1]

orig_lengths = edge_lengths(square)
rot_lengths = edge_lengths(rotated)
scale_lengths = edge_lengths(scaled)

for i, length in enumerate(orig_lengths):
    print(f"边{i}：{length}")
assert [round(x, 6) for x in orig_lengths] == [1, 1, 1, 1]
assert [round(x, 6) for x in rot_lengths] == [1, 1, 1, 1], "旋转后四条边仍应为 1"
assert [round(x, 6) for x in scale_lengths] == [2, 0.5, 2, 0.5], "缩放后边长应发生变化"

print("练习 1 通过：你已经能把 matrix 批量应用到多个点上。")
print("旋转后的点：", [[round(x, 4) for x in p] for p in rotated])
print("缩放后的点：", [[round(x, 4) for x in p] for p in scaled])
print("剪切后的点：", [[round(x, 4) for x in p] for p in sheared])

边0：1.0
边1：1.0
边2：1.0
边3：1.0
练习 1 通过：你已经能把 matrix 批量应用到多个点上。
旋转后的点： [[0.0, 0.0], [0.7071, 0.7071], [0.0, 1.4142], [-0.7071, 0.7071]]
缩放后的点： [[0, 0.0], [2, 0.0], [2, 0.5], [0, 0.5]]
剪切后的点： [[0.0, 0], [1.0, 0], [1.5, 1], [0.5, 1]]


### 练习 2（进阶选做）：比较 transformation 的先后顺序

这题只研究一件事：**matrix 相乘的顺序不同，最后点的位置通常不同**。

不要急着记公式。你只要按照“先发生的变换写在右边”这个规则，把两个组合 matrix 做出来，然后应用到同一个点上。

**你要实现：**

```text
compare_transform_order(point) -> result
```

返回一个字典，包含：

```text
{
    "rotate_then_scale": 先旋转再缩放后的点,
    "scale_then_rotate": 先缩放再旋转后的点,
    "same": 两个结果是否几乎相同
}
```

**步骤提示：**

1. 构造一个 90° 旋转 matrix。
2. 构造一个缩放 matrix，x 方向放大、y 方向压缩。
3. 想清楚：如果要“先旋转，再缩放”，组合 matrix 应该怎么乘？
4. 想清楚：如果要“先缩放，再旋转”，组合 matrix 应该怎么乘？
5. 分别把两个组合 matrix 应用到同一个 `point`。
6. 用一个很小的容差比较两个结果是否相同。

**你应该观察到：**

同一个点、同两个变换，只要顺序换了，结果就可能完全不同。

In [93]:
import math


def compare_transform_order(point):
    """比较“先旋转再缩放”和“先缩放再旋转”的结果。

    point: 一个 2D 点，例如 [1.0, 0.0]
    返回字典：
      - rotate_then_scale: 先旋转再缩放后的点
      - scale_then_rotate: 先缩放再旋转后的点
      - same: 两个结果是否几乎相同

    可以使用前面已经实现好的 rotation_2d、scaling_2d、mat_mul、mat_vec_mul。
    """
    # 组合变换的思路，可以是先把两个矩阵相乘得到一个组合矩阵，再用该组合矩阵应用到点上
    R = rotation_2d(math.pi / 2)
    S = scaling_2d(2, 0.5)
    rotate_then_scale = mat_vec_mul(mat_mul(S, R), point)
    scale_then_rotate = mat_vec_mul(mat_mul(R, S), point)
    same = abs(rotate_then_scale[0] - scale_then_rotate[0]) < 1e-9 and abs(rotate_then_scale[1] - scale_then_rotate[1]) < 1e-9
    return {
        "rotate_then_scale" : rotate_then_scale,
        "scale_then_rotate" : scale_then_rotate,
        "same" : same
    }
    

    

In [94]:
# === 练习 2 测试（不要修改）===
# 这题的重点：同一个点、同两个变换，顺序不同，结果不同。

def _point_close(p, q, tol=1e-9):
    return all(abs(p[i] - q[i]) < tol for i in range(2))

result = compare_transform_order([1.0, 0.0])
print(f"{result}")

assert set(result.keys()) == {"rotate_then_scale", "scale_then_rotate", "same"}
assert _point_close(result["rotate_then_scale"], [0.0, 0.5])
assert _point_close(result["scale_then_rotate"], [0.0, 2.0])
assert result["same"] is False

# 再换一个点，确认函数不是只为 [1,0] 硬编码。
result2 = compare_transform_order([1.0, 1.0])
assert _point_close(result2["rotate_then_scale"], [-2.0, 0.5])
assert _point_close(result2["scale_then_rotate"], [-0.5, 2.0])
assert result2["same"] is False

print("练习 2 通过：你已经亲手验证 transformation 的顺序会改变结果。")
print("[1,0] 先旋转再缩放：", [round(x, 4) for x in result["rotate_then_scale"]])
print("[1,0] 先缩放再旋转：", [round(x, 4) for x in result["scale_then_rotate"]])

{'rotate_then_scale': [1.2246467991473532e-16, 0.5], 'scale_then_rotate': [1.2246467991473532e-16, 2.0], 'same': False}
练习 2 通过：你已经亲手验证 transformation 的顺序会改变结果。
[1,0] 先旋转再缩放： [0.0, 0.5]
[1,0] 先缩放再旋转： [0.0, 2.0]


### 练习 3（挑战选做）：用 eigenvalue 做一个简单判断

这题先不要求你重写 eigenvalue 算法。你可以直接使用前面 Build It 里已经写好的 `eigenvalues_2x2(A)`。

我们只做一个简单判断：如果一个 matrix 不断重复作用在向量上，它大致会让向量越来越大、越来越小，还是保持差不多的尺度？

判断方法很直观：

1. 算出两个 eigenvalue。
2. 看每个 eigenvalue 的绝对值 `abs(lambda)`。
3. 取最大的那个绝对值。
4. 如果它大于 1，重复很多次后通常会放大。
5. 如果它小于 1，重复很多次后通常会缩小。
6. 如果它接近 1，就认为尺度大致稳定。

**你要实现：**

```text
classify_by_eigenvalues(A) -> result
```

返回一个字典：

```text
{
    "eigenvalues": 两个 eigenvalue,
    "max_abs": 两个绝对值里较大的那个,
    "behavior": "放大" 或 "缩小" 或 "稳定"
}
```

**步骤提示：**

1. 调用 `eigenvalues_2x2(A)`。
2. 对两个 eigenvalue 分别取 `abs(...)`。
3. 找到更大的绝对值。
4. 用 `1e-9` 做容差，判断它和 1 的关系。
5. 组装成字典返回。

这题的目标不是证明，而是先把 eigenvalue 和“长期会怎样”建立一个代码上的联系。

In [99]:
def classify_by_eigenvalues(A):
    """用 eigenvalue 的绝对值粗略判断重复作用 A 的长期行为。

    A: 2×2 matrix。
    返回字典：
      - eigenvalues: eigenvalues_2x2(A) 的结果
      - max_abs: 两个 eigenvalue 绝对值里的较大者
      - behavior: "放大" / "缩小" / "稳定"

    可以直接使用前面 Build It 中已经写好的 eigenvalues_2x2(A)。
    """
    eigenvalues = eigenvalues_2x2(A)
    max_abs = max(abs(eigenvalues[0]), abs(eigenvalues[1]))
    if abs(max_abs - 1) > 1e-5 and max_abs > 1:
        behavior = "放大"
    elif abs(max_abs - 1) > 1e-5 and max_abs < 1:
        behavior = "缩小"
    else:
        behavior = "稳定"
    return {
        "eigenvalues" : eigenvalues,
        "max_abs" : max_abs,
        "behavior" : behavior
    }

In [100]:
# === 练习 3 测试（不要修改）===
# 这题只检查你是否会用 eigenvalue 做简单判断。

def _close_number(x, y, tol=1e-9):
    return abs(x - y) < tol

case_grow = classify_by_eigenvalues([[2, 0], [0, 0.5]])
assert case_grow["behavior"] == "放大"
assert _close_number(case_grow["max_abs"], 2.0)

case_shrink = classify_by_eigenvalues([[0.5, 0], [0, 0.25]])
assert case_shrink["behavior"] == "缩小"
assert _close_number(case_shrink["max_abs"], 0.5)

case_stable = classify_by_eigenvalues([[0, -1], [1, 0]])
assert case_stable["behavior"] == "稳定"
assert _close_number(case_stable["max_abs"], 1.0)

case_real = classify_by_eigenvalues([[2, 1], [1, 2]])
assert case_real["behavior"] == "放大"
assert _close_number(case_real["max_abs"], 3.0)

for result in [case_grow, case_shrink, case_stable, case_real]:
    assert set(result.keys()) == {"eigenvalues", "max_abs", "behavior"}
    assert len(result["eigenvalues"]) == 2

print("练习 3 通过：你已经会用 eigenvalue 粗略判断重复作用后的趋势。")
print("放大例子：", case_grow)
print("缩小例子：", case_shrink)
print("稳定例子：", case_stable)

练习 3 通过：你已经会用 eigenvalue 粗略判断重复作用后的趋势。
放大例子： {'eigenvalues': (2.0, 0.5), 'max_abs': 2.0, 'behavior': '放大'}
缩小例子： {'eigenvalues': (0.5, 0.25), 'max_abs': 0.5, 'behavior': '缩小'}
稳定例子： {'eigenvalues': (1j, -1j), 'max_abs': 1.0, 'behavior': '稳定'}


### 简单验证

运行一个最小检查，确认所有语法正确、函数可调用。

In [101]:
# 只跑一个简单例子，检查没有语法错误
import math

# 测试旋转
R = rotation_2d(math.pi / 4)
p = mat_vec_mul(R, [1.0, 0.0])
print("旋转 (1,0) 45°:", [round(x, 4) for x in p])

# 测试 eigenvalues
A = [[4, 2], [1, 3]]
vals = eigenvalues_2x2(A)
print(f"eigenvalues of [[4,2],[1,3]]: {vals[0]:.1f}, {vals[1]:.1f}")

# 测试 determinant
print(f"det(旋转 45°) = {det_2x2(R):.4f}")

print("\n所有基本检查通过 ✅")

旋转 (1,0) 45°: [0.7071, 0.7071]
eigenvalues of [[4,2],[1,3]]: 5.0, 2.0
det(旋转 45°) = 1.0000

所有基本检查通过 ✅


---

## 讨论笔记 💬

学习过程中产生的关键理解：

### $A - \lambda I$ 是什么？

它是一个**临时的"探测器"**。从 $A\mathbf{v} = \lambda\mathbf{v}$ 移项得到 $(A-\lambda I)\mathbf{v} = \mathbf{0}$，故意让 $A-\lambda I$ 压缩一维（奇异），只为找到让条件成立的 $\lambda$。找到后探测器就扔了。真正对空间做变换的是 $A$，不是 $A-\lambda I$。

### 正向路径 vs 反向路径

- **真实 ML 工作（正向）：** 有矩阵 $A$ → 求 eigen → 理解 $A$ 的行为（PCA、稳定性分析等）
- **考试题（反向）：** 给 eigen → 还原矩阵 $A$（仅用于检验理解 $A=VDV^{-1}$）

你的笔记本和未来的 ML 工作永远是正向路径。

### Eigenvector 有无数个

数学定义只要求 $\mathbf{v} \neq \mathbf{0}$ 且满足 $A\mathbf{v} = \lambda\mathbf{v}$。整条线上的向量都满足。代码返回归一化（长度=1）的向量只是**约定**，方便使用。

### 为什么 $\det(A-\lambda I)$ 一定是 $n$ 次多项式？

$\lambda$ 只出现在 $A-\lambda I$ 的对角线上。$n!$ 项行列式展开中，只有**恒等排列**（取全部 $n$ 个对角元素）能产生 $\lambda^n$ 项。其他排列至少跳过一个对角位置，最高次 $\le n-1$。所以它一定是 $n$ 次多项式 → $n$ 个根 → $n$ 个 eigenvalue。

### Eigenvector = 变换的指纹

就像指纹唯一识别一个人，eigenvalue + eigenvector 唯一识别一个变换。知道 $V$ 和 $D$ 就能完整重建 $A = VDV^{-1}$。这就是为什么 PCA 用 eigenvector 做主成分、谱聚类用 eigenvector 揭示图结构——eigen 系统就是数据本身的"指纹"。

---

## 产出

这节课为 PCA（Phase 2）和神经网络 weight 分析建立了**几何基础**。此处的 eigenvalue/eigenvector 代码与生产 ML 系统中驱动降维、谱聚类和稳定性分析的算法是**同一回事**。

核心收获：
1. **Matrix = 空间变换器**。每个 2×2 matrix 定义了 basis vector 的去向，一切随之确定
2. **旋转、缩放、剪切、反射** = 四种基本变换，理解它们你就理解了所有线性变换
3. **组合 = 矩阵乘法**，顺序至关重要（不可交换）
4. **Eigenvalue 告诉你"沿某个方向拉伸多少"**，这是 PCA / RNN 稳定性 / 谱聚类的核心
5. **Determinant = 体积缩放因子**，为 0 意味着不可逆（信息丢失）

---

## 关联

本文的每个概念都与现代 AI 的具体部分相连：

| 概念 | 出现的地方 |
|------|-----------|
| 旋转/缩放/剪切/反射 | 数据增强（随机旋转、缩放、翻转图像），3D 图形变换 |
| 组合变换 | 神经网络中的连续层（每一层是一次矩阵变换） |
| Eigenvalue & Eigenvector | PCA 降维（主成分 = covariance matrix 的 eigenvector） |
| 特征分解 | 谱聚类（图 Laplacian 的 eigenvector），PageRank（邻接矩阵的 eigenvector） |
| Determinant | 判断矩阵是否可逆（奇异检测），概率密度变换（Jacobian determinant） |
| 复数 eigenvalue | RNN 稳定性分析，动力系统的振荡行为 |
| 3D 旋转矩阵 | 计算机视觉中的相机姿态估计，3D 物体渲染 |